# Model Testing Notebook

Test all models with small inputs to verify:
1. All models run correctly
2. Adaptive halting works (UT, UT-L1, UT-L2, TRM)
3. `set_full_compute(True)` forces max steps

In [1]:
import sys
sys.path.insert(0, '..')

import torch
from models import GPTBase, GPTLevel1, GPTLevel2, UT, UTLevel1, UTLevel2, TRM

# Small config for testing
CONFIG = dict(
    vocab_size=100,
    block_size=32,
    n_embd=64,
    n_head=4,
    n_layer=4,
    dropout=0.0,
    lr=1e-3,
)

# Test input
B, T = 2, 16  # batch_size=2, seq_len=16
x = torch.randint(0, CONFIG['vocab_size'], (B, T))
y = torch.randint(0, CONFIG['vocab_size'], (B, T))

print(f"Input shape: {x.shape}")
print(f"Target shape: {y.shape}")

Input shape: torch.Size([2, 16])
Target shape: torch.Size([2, 16])


## 1. Test Fixed-Compute Models (GPT variants)

In [2]:
print("=" * 60)
print("GPT Base")
print("=" * 60)

gpt = GPTBase(**CONFIG)
gpt.eval()

with torch.no_grad():
    logits, loss = gpt(x, y)
    
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"Params: {sum(p.numel() for p in gpt.parameters()):,}")

GPT Base


Logits shape: torch.Size([2, 16, 100])
Loss: 4.6383
Params: 214,244


In [3]:
print("=" * 60)
print("GPT Level 1 (weight tying)")
print("=" * 60)

gpt_l1 = GPTLevel1(**CONFIG)
gpt_l1.eval()

with torch.no_grad():
    logits, loss = gpt_l1(x, y)
    
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"Params: {sum(p.numel() for p in gpt_l1.parameters()):,}")

GPT Level 1 (weight tying)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.6011
Params: 64,868


In [4]:
print("=" * 60)
print("GPT Level 2 (+ step embedding)")
print("=" * 60)

gpt_l2 = GPTLevel2(**CONFIG)
gpt_l2.eval()

with torch.no_grad():
    logits, loss = gpt_l2(x, y)
    
print(f"Logits shape: {logits.shape}")
print(f"Loss: {loss.item():.4f}")
print(f"Params: {sum(p.numel() for p in gpt_l2.parameters()):,}")

GPT Level 2 (+ step embedding)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.5933
Params: 65,124


## 2. Test Adaptive Models - Default Behavior (may halt early)

In [5]:
def test_adaptive_model(model, name, x, y):
    """Test an adaptive model and report halting behavior."""
    print("=" * 60)
    print(f"{name}")
    print("=" * 60)
    
    model.eval()
    
    # Default behavior (may halt early)
    model.set_full_compute(False)
    with torch.no_grad():
        logits, loss = model(x, y)
    
    print(f"Logits shape: {logits.shape}")
    print(f"Loss: {loss.item():.4f}")
    print(f"Params: {sum(p.numel() for p in model.parameters()):,}")
    
    # Check for steps/ponder info
    if hasattr(model, '_last_steps'):
        print(f"Steps taken (default): {model._last_steps} / {model.max_act_steps}")
    if hasattr(model, '_last_ponder_cost'):
        print(f"Ponder cost: {model._last_ponder_cost:.4f}")
    
    return model

In [6]:
ut_config = {**CONFIG, 'ponder_cost_weight': 0.01, 'max_act_steps': 4}
ut = UT(**ut_config)
ut = test_adaptive_model(ut, "UT (Graves' ACT)", x, y)

UT (Graves' ACT)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.6729
Params: 65,189
Ponder cost: 2.6718


In [7]:
ut_l1 = UTLevel1(**ut_config)
ut_l1 = test_adaptive_model(ut_l1, "UT Level 1 (+ dual stream)", x, y)

UT Level 1 (+ dual stream)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.6754
Params: 65,317
Ponder cost: 2.4952


In [8]:
ut_l2_config = {**ut_config, 'n_inner_loops': 2, 'n_outer_loops': 2}
ut_l2 = UTLevel2(**ut_l2_config)
ut_l2 = test_adaptive_model(ut_l2, "UT Level 2 (+ H×L recurrence)", x, y)

UT Level 2 (+ H×L recurrence)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.6237
Params: 65,317
Ponder cost: 2.5347


In [9]:
trm_config = {**ut_l2_config, 'halt_exploration_prob': 0.25}
trm = TRM(**trm_config)
trm = test_adaptive_model(trm, "TRM (Q-based halting)", x, y)

TRM (Q-based halting)
Logits shape: torch.Size([2, 16, 100])
Loss: 4.6273
Params: 65,317
Steps taken (default): 4 / 4


## 3. Test `set_full_compute(True)` - Force Max Steps

In [10]:
def test_full_compute(model, name, x, y):
    """Test model with forced full compute."""
    print("=" * 60)
    print(f"{name} - FULL COMPUTE")
    print("=" * 60)
    
    model.eval()
    model.set_full_compute(True)
    
    with torch.no_grad():
        logits, loss = model(x, y)
    
    print(f"Loss: {loss.item():.4f}")
    
    if hasattr(model, '_last_steps'):
        print(f"Steps taken (full compute): {model._last_steps} / {model.max_act_steps}")
    if hasattr(model, '_last_ponder_cost'):
        print(f"Ponder cost: {model._last_ponder_cost:.4f}")
    
    # Reset to default
    model.set_full_compute(False)
    return model

In [11]:
ut = test_full_compute(ut, "UT", x, y)

UT - FULL COMPUTE
Loss: 4.6729
Ponder cost: 2.6718


In [12]:
ut_l1 = test_full_compute(ut_l1, "UT Level 1", x, y)

UT Level 1 - FULL COMPUTE
Loss: 4.6754
Ponder cost: 2.4952


In [13]:
ut_l2 = test_full_compute(ut_l2, "UT Level 2", x, y)

UT Level 2 - FULL COMPUTE
Loss: 4.6237
Ponder cost: 2.5347


In [14]:
trm = test_full_compute(trm, "TRM", x, y)

TRM - FULL COMPUTE
Loss: 4.6273
Steps taken (full compute): 4 / 4


## 4. Compare Default vs Full Compute Side-by-Side

In [15]:
def compare_modes(model, name, x, y):
    """Compare default and full compute modes."""
    model.eval()
    
    # Default mode
    model.set_full_compute(False)
    with torch.no_grad():
        _, loss_default = model(x, y)
    steps_default = getattr(model, '_last_steps', 'N/A')
    
    # Full compute mode
    model.set_full_compute(True)
    with torch.no_grad():
        _, loss_full = model(x, y)
    steps_full = getattr(model, '_last_steps', 'N/A')
    
    max_steps = getattr(model, 'max_act_steps', 'N/A')
    
    # Reset
    model.set_full_compute(False)
    
    return {
        'Model': name,
        'Max Steps': max_steps,
        'Steps (default)': steps_default,
        'Steps (full)': steps_full,
        'Loss (default)': f"{loss_default.item():.4f}",
        'Loss (full)': f"{loss_full.item():.4f}",
    }

In [16]:
# Recreate models for clean comparison
models_to_test = [
    (UT(**ut_config), "UT"),
    (UTLevel1(**ut_config), "UT-L1"),
    (UTLevel2(**ut_l2_config), "UT-L2"),
    (TRM(**trm_config), "TRM"),
]

results = [compare_modes(m, name, x, y) for m, name in models_to_test]

# Display as table
import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

Model  Max Steps Steps (default) Steps (full) Loss (default) Loss (full)
   UT          4             N/A          N/A         4.6862      4.6862
UT-L1          4             N/A          N/A         4.6775      4.6775
UT-L2          4             N/A          N/A         4.6544      4.6544
  TRM          4               4            4         4.6151      4.6151


## 5. Test Training Mode (exploration affects TRM)

In [17]:
print("Testing TRM in training mode (exploration enabled)")
print("=" * 60)

trm_train = TRM(**trm_config)
trm_train.train()  # Training mode
trm_train.set_full_compute(False)

# Run multiple times to see exploration variance
steps_list = []
for i in range(10):
    with torch.no_grad():
        _, _ = trm_train(x, y)
    steps_list.append(trm_train._last_steps)

print(f"Steps over 10 runs: {steps_list}")
print(f"Mean: {sum(steps_list)/len(steps_list):.2f}, Min: {min(steps_list)}, Max: {max(steps_list)}")

Testing TRM in training mode (exploration enabled)


Steps over 10 runs: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
Mean: 4.00, Min: 4, Max: 4


## 6. Compute Summary

In [18]:
from models.common.compute import calculate_block_passes, get_compute_summary

print("Block passes per forward (max):")
print("=" * 60)

for name in ['gpt', 'gpt_level1', 'gpt_level2', 'ut', 'ut_level1', 'ut_level2', 'trm']:
    summary = get_compute_summary(
        name, 
        n_layer=CONFIG['n_layer'],
        n_inner_loops=2,
        n_outer_loops=2,
    )
    print(summary)

Block passes per forward (max):
gpt: 4 block passes (n_layer=4)
gpt_level1: 4 block passes (n_layer=4)
gpt_level2: 4 block passes (n_layer=4)
ut: 4 block passes (n_layer=4 (max ACT steps))
ut_level1: 8 block passes (n_layer=4 × 2 passes/step)
ut_level2: 24 block passes (n_layer=4 × 2 outer × (2+1) inner)
trm: 24 block passes (n_layer=4 × 2 outer × (2+1) inner)
